In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis
from scipy.fft import fft, fftfreq
from statsmodels.graphics.tsaplots import plot_acf

df = pd.read_csv('merged_data.txt', sep='\t')
df.columns = [str(col) for col in df.columns]

# Denoising
df_clean = df.rolling(window=5).mean()
df_clean = df_clean.bfill()

df_clean.to_csv('clean_data.txt', sep='\t', index=False)

In [2]:
official_ranges = {
    "p1_juice_inlet": (0, 1000),      
    "p2_juice_outlet": (0, 1000),     
    "t_juice_outlet": (50, 150),    
    "f_juice_flow_evap1_in": (0, 500),
    "lc51_03cv": (0, 100),            
    "lc51_03x": (0, 100),           
    "lc51_03pv": (0, 100),            
    "t_juice_evap1_in": (50, 150),   
    "t_juice_evap1_out": (50, 150),   
    "d_juice_evap1_in": (0, 25),      
    "d_juice_evap1_out": (13, 41),    
    "steam_flow": (1, 100),           
    "steam_pressure": (100, 300),     
    "steam_temp": (50, 150),         
    "vapour_pressure": (0, 250),      
    "vapour_temp": (50, 150),        
    "p1_juice_inlet_act2": (0, 1000), 
    "p2_juice_outlet_act2": (0, 1000),
    "t_juice_inlet_act2": (0, 150),  
    "fc57_03pv": (0, 100),           
    "fc57_03cv": (0, 100),            
    "fc57_03x": (0, 100),            
    "p1_water_inlet_act3": (0, 4000), 
    "p2_water_outlet_act3": (0, 4000),
    "t_water_outlet_act3": (0, 150),  
    "f_water_flow_boiler_in": (0, 40), 
    "lc74_20cv": (0, 100),            
    "lc74_20x": (0, 100),             
    "lc74_20pv": (0, 100),            
    "f_steam_boiler_out": (0, 40),    
    "p_steam_boiler_out": (0, 4000),  
    "t_steam_boiler_out": (0, 550)    
}

def check_outliers(df, ranges):
    outlier_summary = []
    for col, (min_val, max_val) in ranges.items():
        if col in df.columns:
            outliers = df[(df[col] < min_val) | (df[col] > max_val)]
            count = len(outliers)
            if count > 0:
                outlier_summary.append({
                    "Sensor": col,
                    "Outlier Count": count,
                    "Min Found": df[col].min(),
                    "Max Found": df[col].max(),
                    "Expected Range": f"{min_val}-{max_val}"
                })
    return pd.DataFrame(outlier_summary)

summary = check_outliers(df_clean, official_ranges)
print("Outlier Analysis Summary:")
print(summary)

Outlier Analysis Summary:
Empty DataFrame
Columns: []
Index: []


In [3]:
print(f"Dataset Shape: {df.shape}") 

# Check for Nulls
print("\nMissing values per column:")
print(df[['p1_juice_inlet', 'vapour_pressure', 'vapour_temp']].isnull().sum())

# Check descriptive stats
print("\nDescriptive Statistics for your sensors:")
print(df[['p1_juice_inlet', 'vapour_pressure', 'vapour_temp']].describe())

Dataset Shape: (2160000, 33)

Missing values per column:
p1_juice_inlet     0
vapour_pressure    0
vapour_temp        0
dtype: int64

Descriptive Statistics for your sensors:
       p1_juice_inlet  vapour_pressure   vapour_temp
count    2.160000e+06     2.160000e+06  2.160000e+06
mean     6.453046e+02     1.684285e+02  1.329577e+02
std      3.722317e+01     6.794530e+00  9.589832e-01
min      3.817000e+02     1.260000e+02  1.266000e+02
25%      6.283000e+02     1.647000e+02  1.325000e+02
50%      6.469000e+02     1.691000e+02  1.331000e+02
75%      6.657000e+02     1.730000e+02  1.336000e+02
max      8.129000e+02     1.867000e+02  1.360000e+02
